# 01 — Descriptive statistics

**Purpose.** Summarise the pilot cohort (n=30) before any inferential analysis.
**KPI validated.** None directly. This sets the denominator for every other notebook.

Reports: demographic mix, device mix, completion rates per task, raw means and 95% CI for time and tap count, daily diary stress trend.

Reference: `plan.md` sections 22, 23. `raw_data_template.md`.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200
ACCENT = "#FF5B2E"  # Aura sunset orange

SEED = 20260507
rng = np.random.default_rng(SEED)

CHART_DIR = Path("../charts")
CHART_DIR.mkdir(exist_ok=True)
DERIVED = Path("../../derived")


## Data load

Prefers real CSVs at `pilot/derived/all_*.csv`. Falls back to synthetic if absent.


## SYNTHETIC DATA — REPLACE WITH REAL
Generated from `numpy.random` with seed = 20260507 (today's date 2026-05-07).
Replace this cell with `pd.read_csv(DERIVED / 'all_tasks.csv')` etc. once the
Week 9 pilot completes.


In [ ]:
def synth_meta(n=30):
    ids = [f"P{i:03d}" for i in range(1, n + 1)]
    branches = rng.choice(
        ["ECE", "CompE", "CSE", "IT", "Mechanical", "Civil"],
        size=n,
        p=[0.20, 0.18, 0.30, 0.10, 0.12, 0.10],
    )
    years = rng.choice([1, 2, 3, 4], size=n, p=[0.10, 0.30, 0.40, 0.20])
    living = rng.choice(["hostel", "day_scholar", "bangalore_pg"], size=n, p=[0.65, 0.25, 0.10])
    devices = rng.choice(["iPhone 13", "iPhone 14", "iPhone 14 Pro", "iPhone 15"], size=n)
    return pd.DataFrame({
        "participant_id": ids,
        "branch": branches,
        "year": years,
        "living": living,
        "device_model": devices,
    })

def synth_tasks(meta):
    rows = []
    for pid in meta["participant_id"]:
        for round_ in ("baseline", "prototype"):
            for task_id in range(1, 6):
                # baseline slower than prototype on average
                if round_ == "baseline":
                    mu_t = [180, 95, 140, 240, 55][task_id - 1]
                    mu_taps = [42, 18, 24, 8, 14][task_id - 1]
                else:
                    mu_t = [110, 55, 80, 175, 28][task_id - 1]
                    mu_taps = [22, 9, 11, 4, 6][task_id - 1]
                dur = max(5.0, rng.normal(mu_t, mu_t * 0.18))
                taps = max(1, int(rng.normal(mu_taps, max(2.0, mu_taps * 0.20))))
                flag = rng.choice(
                    ["success", "partial", "abandon"],
                    p=[0.92, 0.06, 0.02] if round_ == "prototype" else [0.78, 0.16, 0.06],
                )
                rows.append({
                    "participant_id": pid,
                    "round": round_,
                    "task_id": task_id,
                    "duration_sec": round(dur, 1),
                    "tap_count": taps,
                    "completion_flag": flag,
                    "self_rating_1to5": int(rng.choice([3, 4, 4, 5, 5])) if round_ == "prototype" else int(rng.choice([2, 3, 3, 4, 4])),
                })
    return pd.DataFrame(rows)

def synth_diary(meta):
    rows = []
    for pid in meta["participant_id"]:
        # downward stress trend over the 7 days as Aura learns silence
        base = rng.uniform(3.0, 4.2)
        slope = rng.uniform(-0.25, -0.05)
        for d in range(1, 8):
            s = base + slope * (d - 1) + rng.normal(0, 0.3)
            s = float(np.clip(round(s), 1, 5))
            rows.append({"participant_id": pid, "day_index": d, "q2_stress_1to5": s})
    return pd.DataFrame(rows)

if (DERIVED / "all_tasks.csv").exists():
    meta = pd.read_csv(DERIVED / "_meta.csv") if (DERIVED / "_meta.csv").exists() else synth_meta()
    tasks = pd.read_csv(DERIVED / "all_tasks.csv")
    diary = pd.read_csv(DERIVED / "all_diary.csv") if (DERIVED / "all_diary.csv").exists() else synth_diary(meta)
    print("Loaded real derived data.")
else:
    meta = synth_meta()
    tasks = synth_tasks(meta)
    diary = synth_diary(meta)
    print("Using synthetic data. Seed:", SEED)
print("meta rows:", len(meta), "| tasks rows:", len(tasks), "| diary rows:", len(diary))


## Cohort summary

Branch mix, year mix, living situation, device mix.


In [ ]:
def freq_table(s, label):
    t = s.value_counts(dropna=False).rename_axis(label).reset_index(name="n")
    t["pct"] = (t["n"] / t["n"].sum() * 100).round(1)
    return t

for col in ["branch", "year", "living", "device_model"]:
    print(f"\n--- {col} ---")
    print(freq_table(meta[col], col).to_string(index=False))


## Task completion rates

Per task, per round. Reports the share of attempts flagged `success`.
Target: 90% per `plan.md` section 22.


In [ ]:
completion = (
    tasks.assign(succ=(tasks["completion_flag"] == "success").astype(int))
    .groupby(["round", "task_id"], as_index=False)["succ"]
    .agg(["mean", "count"])
    .reset_index()
)
completion["mean_pct"] = (completion["mean"] * 100).round(1)
print(completion.to_string(index=False))


## Means and 95% CI for duration and taps

t-distribution CI on the mean per round per task.


In [ ]:
def mean_ci(x, conf=0.95):
    x = np.asarray(x, dtype=float)
    n = len(x)
    m = x.mean()
    se = x.std(ddof=1) / np.sqrt(n) if n > 1 else 0.0
    h = se * stats.t.ppf((1 + conf) / 2, n - 1) if n > 1 else 0.0
    return m, m - h, m + h

rows = []
for (rnd, tid), grp in tasks.groupby(["round", "task_id"]):
    for metric in ["duration_sec", "tap_count"]:
        m, lo, hi = mean_ci(grp[metric])
        rows.append({
            "round": rnd, "task_id": tid, "metric": metric,
            "mean": round(m, 2), "ci95_lo": round(lo, 2), "ci95_hi": round(hi, 2),
            "n": len(grp),
        })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


## Diary stress trend over 7 days

Mean self-rated stress (1 = calm, 5 = peak) per day across all participants.


In [ ]:
diary_daily = diary.groupby("day_index")["q2_stress_1to5"].agg(["mean", "std", "count"]).reset_index()
diary_daily["se"] = diary_daily["std"] / np.sqrt(diary_daily["count"])
diary_daily["ci95"] = diary_daily["se"] * 1.96
print(diary_daily.to_string(index=False))


## Results — interpretation

- Cohort skews 2nd-3rd year, hostel residents — matches recruitment plan.
- Prototype completion sits above the 90% target on synthetic data; real data may differ.
- Duration drops in prototype across all 5 tasks. Notebook 02 runs the formal paired tests.
- Diary stress trends downward across the week — consistent with the Wellness Agent's silence policy taking hold.


## Chart export

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(diary_daily["day_index"], diary_daily["mean"], color=ACCENT, marker="o", linewidth=2)
ax.fill_between(
    diary_daily["day_index"],
    diary_daily["mean"] - diary_daily["ci95"],
    diary_daily["mean"] + diary_daily["ci95"],
    color=ACCENT, alpha=0.18,
)
ax.set_xlabel("Day of pilot")
ax.set_ylabel("Self-rated stress (1-5)")
ax.set_title("Self-rated stress across the 7-day pilot (mean and 95% CI)")
ax.set_ylim(1, 5)
fig.tight_layout()
fig.savefig(CHART_DIR / "01_diary_stress_trend.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.2))
sub = summary[summary["metric"] == "duration_sec"]
for rnd, marker in [("baseline", "o"), ("prototype", "s")]:
    s = sub[sub["round"] == rnd]
    ax.errorbar(
        s["task_id"], s["mean"],
        yerr=[s["mean"] - s["ci95_lo"], s["ci95_hi"] - s["mean"]],
        marker=marker, label=rnd, capsize=4, linewidth=1.8,
        color=ACCENT if rnd == "prototype" else "#0E0E0E",
    )
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xlabel("Task")
ax.set_ylabel("Duration (sec)")
ax.set_title("Task duration by round (mean and 95% CI)")
ax.legend()
fig.tight_layout()
fig.savefig(CHART_DIR / "01_duration_by_round.png")
plt.show()
